# Lab 09: Student Initialisation, Prune-Then-Distill

**Tier 2 lab.** Part A executes and asserts anywhere, because it performs real surgery on a
real 135M model on CPU; Part B trains only when `RUN_TRAINING = True`.

**The question.** Every student so far started as a pretrained checkpoint someone else made.
But the teacher already *contains* students. Delete some of its layers and what remains is a
smaller model that still remembers most of what the big one knew, because every weight you
kept is unchanged. Deleting whole layers this way is called depth pruning, and it is this
lab's subject. Minitron's result (paper 2407.14679) is the headline: prune a teacher, distill
briefly into the pruned model, and match a from-scratch model of the same size at **a few
percent of its training compute**. Sheared-LLaMA reached similar economics with structured
pruning, which means pruning that removes regular blocks of the network (whole layers, or
whole slices of width) rather than scattered individual weights. The lineage runs back
through DistilBERT, which initialised its student from alternating teacher layers, and
TinyBERT, which trained the student to match the teacher's internal features layer-to-layer.

The comparison that matters is therefore three-way, at a **fixed distillation budget**,
meaning every candidate gets exactly the same amount of training compute after
initialisation:

| init | what it knows at step 0 | cost to obtain |
|---|---|---|
| random | nothing | free |
| pretrained-small (360M) | its own pretraining | someone spent it for you |
| pruned-teacher | most of the teacher, damaged | minutes of surgery |

**The skill this lab teaches is evaluating an initialisation *before* spending the training
budget on it.** That breaks into three pieces: measuring layer importance (which layers the
model can afford to lose, defined precisely in the next section), performing the surgery with
shape discipline, and running a sanity-check protocol on the freshly pruned patient. All of
it executes right here in Part A on a small model, because none of it is training.

In [1]:
import sys, os, json, math, copy
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import shift_for_next_token, masked_mean
from kd_pipeline import set_seed_everywhere, config_fingerprint, MemoryPlan, \
                        full_ft_gb, infer_gb, RunManifest

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Part A · 1: Measure layer importance on a real model

Which layers can go? Ask the model itself: drop each layer in turn, run a probe set, and
record the damage. A probe set is a small fixed batch of held-out examples used only for
measurement, never for training. The damage metric is the increase in masked next-token loss
on that probe set. Next-token loss is the ordinary language-model cross-entropy: the negative
log-probability the model assigns to the token that actually comes next. "Masked" means the
average is taken only over the completion tokens, the positions the completion mask marks as
answer rather than prompt. This increase is what "layer importance" means in this lab: how
much worse the model gets when that one layer is gone. It is the right currency for the job
because it is exactly the quantity distillation training will later try to repair.

This executes here, for real, on `SmolLM2-135M` (30 layers, small enough for CPU): 30
ablations, meaning one measurement per layer with that layer temporarily removed, over a
small probe batch. Three facts to look for in the output. All three are standard across
transformer families, and all three are worth seeing with your own eyes rather than
believing:

1. **The ends matter most.** The first layers lift raw tokens and positions into the model's
   internal representation, and the last layers project that representation back onto the
   vocabulary. Remove either end and the damage is catastrophic.
2. **The middle is soft.** Deep-middle layers each cost little to remove, because they refine
   the representation rather than transform it. This softness is prune-then-distill's entire
   opportunity.
3. **Importance is not uniform even in the middle.** Some middle layers matter noticeably
   more than their neighbours, which is why Minitron *measures* importance instead of
   dropping every other layer the DistilBERT way.

In [2]:
from transformers import AutoModelForCausalLM

PROBE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(PROBE_MODEL, dtype=torch.float32).eval()
n_layers = model.config.num_hidden_layers
print(f"{PROBE_MODEL}: {n_layers} layers, hidden {model.config.hidden_size}")

ev = torch.load("../data/lab03/eval.pt")
probe_ids, probe_mask = ev["input_ids"][:8], ev["mask"][:8]

@torch.no_grad()
def probe_loss(m):
    logits = m(probe_ids).logits
    lp = F.log_softmax(logits[:, :-1].float(), dim=-1)
    nll = -lp.gather(-1, probe_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return float(masked_mean(nll, probe_mask[:, 1:]))

base = probe_loss(model)
layers = model.model.layers
damage = []
for i in range(n_layers):
    kept = torch.nn.ModuleList([l for j, l in enumerate(layers) if j != i])
    model.model.layers = kept
    damage.append(probe_loss(model) - base)
    model.model.layers = layers
    if i % 6 == 0:
        print(f"  ablated layer {i:>2}: +{damage[-1]:.4f} nats")

ranked = sorted(range(n_layers), key=lambda i: damage[i])
ends = {0, 1, n_layers - 2, n_layers - 1}
print(f"\nbase probe loss {base:.4f} | least important: {ranked[:6]} | most: {ranked[-4:]}")
assert all(d > -0.02 for d in damage), "removing a layer should never clearly help"
assert max(damage[i] for i in ends) > 2 * (sum(damage) / len(damage)), \
    "an end layer should be far above average importance"
assert min(damage[i] for i in ends) > min(damage), "the softest layer is not an end layer"
json.dump({"model": PROBE_MODEL, "base_loss": base, "damage": damage},
          open("../data/lab09_importance_135m.json", "w"))
print("importance profile measured on a real model and saved — ends hard, middle soft")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:30,  8.95it/s]

Loading weights:  35%|███▍      | 95/272 [00:00<00:00, 529.83it/s]

Loading weights:  56%|█████▌    | 151/272 [00:00<00:00, 443.03it/s]

...

  ablated layer  6: +0.1896 nats


  ablated layer 12: +0.1290 nats


  ablated layer 18: +0.2323 nats


  ablated layer 24: +0.5055 nats



base probe loss 0.9275 | least important: [5, 4, 12, 3, 7, 9] | most: [29, 11, 1, 0]
importance profile measured on a real model and saved — ends hard, middle soft


## Part A · 2: Surgery, with the checks a surgeon owes the patient

Depth pruning is a state-dict operation. A state dict is PyTorch's dictionary of a model's
weights: it maps parameter names (strings like `model.layers.7.self_attn.q_proj.weight`) to
weight tensors, and it is what gets saved and loaded when you checkpoint a model. The surgery
is: build a smaller config, then copy over the embeddings, the final norm, the output head,
and the **kept** layers, renumbered contiguously so the kept layers fill slots 0, 1, 2 and so
on with no gaps. The traps are all bookkeeping, so the checks are all mechanical, and they
execute here by actually pruning the 135M model to 22 layers using A·1's measured ranking
(30 layers minus the 8 least important is 22):

1. **Config honesty.** `num_hidden_layers` must match the number of layers you actually
   copied. If it does not, loading silently reinitialises the missing ones with fresh random
   weights, which is worse than crashing, because you would then distill into a lie without
   any error message telling you so.
2. **Renumbering.** Kept layer *k* becomes layer *0..n*, in original order. Order matters
   because layers compose: each layer's output is the next layer's input, so changing the
   order changes the computation.
3. **The patient lives.** A forward pass runs, and its loss sits *between* the intact model's
   loss and a random-init model's loss. Something close to gibberish is expected, because
   Minitron's pruned models are also bad before distillation. But a loss equivalent to random
   init means the surgery botched something, because a correct prune keeps real knowledge,
   not just tensor shapes.
4. **Keep the receipts.** Which layers were kept, and which ranking chose them, written into
   the checkpoint dir, so the surgery can be audited later.

In [3]:
from transformers import AutoConfig

def depth_prune(src_model, keep_layers):
    keep_layers = sorted(keep_layers)
    cfg = copy.deepcopy(src_model.config)
    cfg.num_hidden_layers = len(keep_layers)
    pruned = AutoModelForCausalLM.from_config(cfg)
    src, dst = src_model.state_dict(), {}
    for name, tensor in src.items():
        if ".layers." in name:
            lid = int(name.split(".layers.")[1].split(".")[0])
            if lid in keep_layers:
                new_lid = keep_layers.index(lid)
                dst[name.replace(f".layers.{lid}.", f".layers.{new_lid}.")] = tensor
        else:
            dst[name] = tensor
    missing, unexpected = pruned.load_state_dict(dst, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    return pruned

n_drop = 8
drop = set(ranked[:n_drop])                       # measured least-important
keep = [i for i in range(n_layers) if i not in drop]
pruned = depth_prune(model, keep).eval()
rand = AutoModelForCausalLM.from_config(pruned.config).eval()

p_loss, r_loss = probe_loss(pruned), probe_loss(rand)
print(f"intact {base:.3f} | pruned({len(keep)}L) {p_loss:.3f} | random-init {r_loss:.3f} nats")
assert pruned.config.num_hidden_layers == len(keep)
assert base < p_loss < r_loss, "pruned must sit between intact and random"
assert p_loss < 0.7 * r_loss, "surgery kept real knowledge, not just shapes"

# The DistilBERT-style contrast: drop a contiguous mid-block of the same size.
contig = depth_prune(model, [i for i in range(n_layers)
                             if not (n_layers//2 - n_drop//2 <= i < n_layers//2 + n_drop//2)]).eval()
c_loss = probe_loss(contig)
print(f"measured-drop {p_loss:.3f} vs contiguous-mid-drop {c_loss:.3f} "
      f"({'measured wins' if p_loss < c_loss else 'contiguous wins — interesting, keep it'})")
del model, pruned, rand, contig
print("surgery verified: config honest, layers renumbered, patient alive, receipts kept")

intact 0.927 | pruned(22L) 2.766 | random-init 11.268 nats


measured-drop 2.766 vs contiguous-mid-drop 5.519 (measured wins)
surgery verified: config honest, layers renumbered, patient alive, receipts kept


## Part A · 3: The arms and the budget

The Part B experiment, written down and asserted before any of it runs: three inits (the
three arms of the experiment), one teacher (1.7B), one distillation recipe (Lab 04's cached
pipeline, chosen because it is the cheapest and its cache cost is already priced), one
budget. The pruned arm cuts the 1.7B teacher's 24 layers down using the measured ranking to a
patient of roughly 1.0B parameters. The ranking is recomputed on the 1.7B itself, because
A·1's *profile* (the specific per-layer damage numbers) does not transfer across models; only
the *method* of measuring it does.

Note what makes the comparison honest and slightly unfair at the same time:
`pretrained-360M` is a smaller model than `pruned-1.0B`, so the two arms differ in capacity
as well as in initialisation. That is the realistic decision though, because "use the small
sibling checkpoint" versus "prune my teacher" is exactly the choice you face in practice,
capacity difference included. The verdict section says how to read the result with that
difference in mind.

In [4]:
ARMS = {
    "random":     dict(init="random-1.0B",  params_b=1.0),
    "pretrained": dict(init="SmolLM2-360M", params_b=0.36),
    "pruned":     dict(init="prune-1.7B",   params_b=1.0),
}
BUDGET = dict(teacher="HuggingFaceTB/SmolLM2-1.7B-Instruct",
              recipe="lab04-cached-topk64", max_steps=1500, lr=3e-5, seed=SEED)

for name, arm in ARMS.items():
    plan = (MemoryPlan(total_gb=128.0)
            .add(f"student {arm['params_b']}B full FT", full_ft_gb(arm["params_b"]))
            .add("activations + cache reads", 6.0))
    plan.assert_fits()
    print(f"{name:>11}: {arm['init']:<14} {plan.planned_gb:>6.1f} GB planned — fits")
assert len({config_fingerprint({**BUDGET, **a}) for a in ARMS.values()}) == 3
print("\nthree arms, one budget, all within memory — registered")

     random: random-1.0B      22.0 GB planned — fits
 pretrained: SmolLM2-360M     11.8 GB planned — fits
     pruned: prune-1.7B       22.0 GB planned — fits

three arms, one budget, all within memory — registered


## Part B: Prune the 1.7B, then distill three times

The steps, using only machinery this course has already verified: recompute A·1's importance
profile on the 1.7B (same code, bigger probe batch), then `depth_prune` it (the same
function, which works on any transformer of this shape by construction, because it only
manipulates state-dict entries by name), then run Lab 04's stage-2 trainer once per init
against the existing cache. Dropping the 10 least important of the 24 layers leaves 14,
which lands the patient near 1.0B parameters. Nothing below is new except the loop order,
which is the point: by Lab 09 the pipeline is vocabulary, not code.

In [5]:
import glob

def importance_profile(model, probe_ids, probe_mask):
    base = probe_loss_on(model, probe_ids, probe_mask)
    layers = model.model.layers
    out = []
    for i in range(model.config.num_hidden_layers):
        model.model.layers = torch.nn.ModuleList(
            [l for j, l in enumerate(layers) if j != i])
        out.append(probe_loss_on(model, probe_ids, probe_mask) - base)
        model.model.layers = layers
    return out

@torch.no_grad()
def probe_loss_on(m, ids, mask):
    logits = m(ids.to(m.device)).logits
    lp = F.log_softmax(logits[:, :-1].float(), dim=-1)
    nll = -lp.gather(-1, ids[:, 1:].unsqueeze(-1).to(m.device)).squeeze(-1)
    return float(masked_mean(nll, mask[:, 1:].to(m.device)))

if RUN_TRAINING:
    from lab04_stage2 import stage2_train_from_cache      # or inline Lab 04's function
    teacher = AutoModelForCausalLM.from_pretrained(
        BUDGET["teacher"], dtype=torch.bfloat16).to(device).eval()
    ev = torch.load("../data/lab03/eval.pt")
    dmg = importance_profile(teacher, ev["input_ids"][:16], ev["mask"][:16])
    ranked_t = sorted(range(len(dmg)), key=lambda i: dmg[i])
    n_drop_t = 10                                          # 24 -> 14 layers ~ 1.0B
    keep_t = [i for i in range(len(dmg)) if i not in set(ranked_t[:n_drop_t])]
    patient = depth_prune(teacher, keep_t)
    patient.save_pretrained("../runs/lab09/pruned_init")
    json.dump({"kept": keep_t, "damage": dmg},
              open("../runs/lab09/pruned_init/surgery.json", "w"))
    del teacher

    inits = {"random": None, "pretrained": "HuggingFaceTB/SmolLM2-360M-Instruct",
             "pruned": "../runs/lab09/pruned_init"}
    for name, path in inits.items():
        print(f"=== distilling into {name} ===")
        # stage2_train_from_cache(CFG with student=path or from_config for random)
        # -> saves ../runs/lab09/{name}_{fingerprint}
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("Surgery is minutes; each distillation is one Lab 04 stage-2 run.")

RUN_TRAINING=False — Part B compiled but did not execute.
Surgery is minutes; each distillation is one Lab 04 stage-2 run.


## Part C: The verdict

**Expected results, and how to read the unfair capacity difference.**

- At step 0: `pruned` is already far better than `random` (A·2 showed why: the surgery keeps
  real knowledge). It is usually already better than `pretrained` on *teacher-agreement*,
  meaning how often the student's top predicted token matches the teacher's, because the
  pruned model inherits the teacher's habits directly. At the same time it is worse on
  general fluency, because it is damaged.
- At the budget's end: `pruned` ≥ `pretrained` > `random` on agreement and KL (the
  Kullback-Leibler divergence between student and teacher token distributions, the course's
  standard closeness measure) is the Minitron-shaped outcome. `random` closing less than half
  of its initial gap in 1,500 steps is normal. That arm's honest lesson is *how much*
  pretraining a random init is missing.
- The interesting cell is `pruned` vs `pretrained` **per parameter**: if the 1.0B pruned
  model beats the 360M checkpoint, is that because of the init, or because of the extra 640M
  parameters (1.0B minus 360M)? The control experiment that separates those two explanations
  is exercise 1. Without it, claim only "prune-then-distill beat the available alternatives
  at this budget", which is the deployable claim anyway.

**Failure signatures.**

- *Pruned arm trains to gibberish.* Almost always renumbering or config drift. Rerun A·2's
  four checks against the 1.7B surgery artifacts. The `surgery.json` receipt exists exactly
  for this.
- *Pruned no better than random at step 0.* The state dict silently reinitialised, which
  means some weights were dropped on load and replaced with random ones. Check the
  `strict=False` missing/unexpected lists you asserted empty in A·2.
- *`pretrained` wins big.* Legitimate when the teacher and the small sibling were trained on
  the same data recipe, as they were here, since both come from SmolLM2's own family.
  Prune-then-distill shines brightest when no good small sibling *exists*; note that
  condition in the verdict.

**The verdict to write:** for your next real project, which init do you choose at (a) zero
small-checkpoint availability, (b) a good small sibling available, (c) a 10× larger budget?
Three sentences, each citing one number from these runs.

## Exercises

1. **The capacity control.** Prune the 1.7B all the way down to a ~360M-equivalent model.
   Drop layers, and note that Minitron would also thin the width; depth-only pruning gets
   ugly below roughly 50% layer removal, and this exercise lets you observe that. Distill at
   the same budget and complete the per-parameter comparison Part C left open.
2. **Iterative vs one-shot.** Drop 10 layers in one surgery, versus two surgeries of 5 layers
   each with 300 distillation steps between them. Sheared-LLaMA argues for the staged
   approach; measure the difference yourself.
3. **Importance metric ablation.** Rerank layers by weight magnitude (a metric that needs no
   forward passes) instead of measured damage; prune; distill 300 steps. How much does the
   cheap metric cost against the measured one?
4. **Feature matching.** Add a TinyBERT-style hidden-state MSE term (match teacher layer 2k
   to student layer k through a linear projector, a small trainable layer that maps between
   the two hidden sizes) to the pruned arm's loss for the first 300 steps. Does early feature
   guidance speed the repair? (This previews Lab 10's representation section.)